In [2]:
import pandas as pd
from pathlib import Path

data_path = Path("..") / "cleaned_dataset.csv"

df = pd.read_csv(data_path)

print("Dataset loaded successfully!")
print("Rows and columns:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

df.head()

Dataset loaded successfully!
Rows and columns: (51290, 23)

Column names:
['order_id', 'order_date', 'ship_date', 'ship_mode', 'customer_name', 'segment', 'state', 'country', 'market', 'region', 'product_id', 'category', 'sub_category', 'product_name', 'sales', 'quantity', 'discount', 'profit', 'shipping_cost', 'order_priority', 'year', 'shipping_days', 'profit_margin']


,order_id,order_date,ship_date,ship_mode,customer_name,segment,state,country,market,region,...,product_name,sales,quantity,discount,profit,shipping_cost,order_priority,year,shipping_days,profit_margin
0,AG-2011-2040,2011-01-01,2011-01-06,Standard Class,Toby Braunhardt,Consumer,Constantine,Algeria,Africa,Africa,...,"Tenex Lockers, Blue",408.0,2,0.0,106.140,35.46,Medium,2011,5,26.014706
1,IN-2011-47883,2011-01-01,2011-01-08,Standard Class,Joseph Holt,Consumer,New South Wales,Australia,APAC,Oceania,...,"Acme Trimmer, High Speed",120.0,3,0.1,36.036,9.72,Medium,2011,7,30.030000
2,HU-2011-1220,2011-01-01,2011-01-05,Second Class,Annie Thurman,Consumer,Budapest,Hungary,EMEA,EMEA,...,"Tenex Box, Single Width",66.0,4,0.0,29.640,8.17,High,2011,4,44.909091
3,IT-2011-3647632,2011-01-01,2011-01-05,Second Class,Eugene Moren,Home Office,Stockholm,Sweden,EU,North,...,"Enermax Note Cards, Premium",45.0,3,0.5,-26.055,4.82,High,2011,4,-57.900000
4,IN-2011-47883,2011-01-01,2011-01-08,Standard Class,Joseph Holt,Consumer,New South Wales,Australia,APAC,Oceania,...,"Eldon Light Bulb, Duo Pack",114.0,5,0.1,37.770,4.70,Medium,2011,7,33.131579


In [3]:
print("Dataset shape:", df.shape)

print("\nMissing values in each column:")
print(df.isnull().sum())

print("\nTotal missing values:", df.isnull().sum().sum())

print("\nDuplicate rows:", df.duplicated().sum())

print("\nData types:")
print(df.dtypes)

Dataset shape: (51290, 23)

Missing values in each column:
order_id          0
order_date        0
ship_date         0
ship_mode         0
customer_name     0
segment           0
state             0
country           0
market            0
region            0
product_id        0
category          0
sub_category      0
product_name      0
sales             0
quantity          0
discount          0
profit            0
shipping_cost     0
order_priority    0
year              0
shipping_days     0
profit_margin     0
dtype: int64

Total missing values: 0

Duplicate rows: 0

Data types:
order_id           object
order_date         object
ship_date          object
ship_mode          object
customer_name      object
segment            object
state              object
country            object
market             object
region             object
product_id         object
category           object
sub_category       object
product_name       object
sales             float64
quantity            i

In [4]:
# Create a copy so the original dataset remains unchanged
df_clean = df.copy()

# Remove duplicate rows
df_clean = df_clean.drop_duplicates()

# Convert date columns to datetime
df_clean["order_date"] = pd.to_datetime(
    df_clean["order_date"], errors="coerce"
)
df_clean["ship_date"] = pd.to_datetime(
    df_clean["ship_date"], errors="coerce"
)

# Convert numeric columns safely
numeric_columns = [
    "sales", "quantity", "discount", "profit",
    "shipping_cost", "shipping_days", "profit_margin"
]

for column in numeric_columns:
    df_clean[column] = pd.to_numeric(
        df_clean[column], errors="coerce"
    )

# Remove rows with invalid essential values
df_clean = df_clean.dropna(
    subset=["order_id", "order_date", "sales", "profit"]
)

print("Original rows:", len(df))
print("Cleaned rows:", len(df_clean))
print("Rows removed:", len(df) - len(df_clean))
print("Remaining missing values:", df_clean.isnull().sum().sum())
print("Remaining duplicate rows:", df_clean.duplicated().sum())

Original rows: 51290
Cleaned rows: 51290
Rows removed: 0
Remaining missing values: 0
Remaining duplicate rows: 0


In [5]:
from pathlib import Path

output_path = Path("cleaned_customer_data.csv")

df_clean.to_csv(output_path, index=False)

print("Cleaned dataset saved successfully!")
print("File location:", output_path.resolve())
print("Final dataset shape:", df_clean.shape)

Cleaned dataset saved successfully!
File location: C:\Users\arjun\OneDrive\Desktop\ABTalks\Day48\cleaned_customer_data.csv
Final dataset shape: (51290, 23)


In [6]:
# Create order month and day features
df_clean["order_month"] = df_clean["order_date"].dt.month
df_clean["order_day"] = df_clean["order_date"].dt.day
df_clean["order_day_of_week"] = df_clean["order_date"].dt.dayofweek

# Create a sales category
df_clean["sales_category"] = pd.cut(
    df_clean["sales"],
    bins=[-float("inf"), 100, 500, float("inf")],
    labels=["Low", "Medium", "High"]
)

# Create a profit status
df_clean["profit_status"] = df_clean["profit"].apply(
    lambda x: "Profit" if x > 0 else ("Loss" if x < 0 else "Break-even")
)

print("Feature engineering completed!")
print("\nNew columns:")
print(["order_month", "order_day", "order_day_of_week",
       "sales_category", "profit_status"])

print("\nUpdated dataset shape:", df_clean.shape)
print("\nPreview:")
display(df_clean[[
    "order_date", "sales", "profit",
    "order_month", "order_day_of_week",
    "sales_category", "profit_status"
]].head())

Feature engineering completed!

New columns:
['order_month', 'order_day', 'order_day_of_week', 'sales_category', 'profit_status']

Updated dataset shape: (51290, 28)

Preview:


,order_date,sales,profit,order_month,order_day_of_week,sales_category,profit_status
0,2011-01-01,408.0,106.140,1,5,Medium,Profit
1,2011-01-01,120.0,36.036,1,5,Medium,Profit
2,2011-01-01,66.0,29.640,1,5,Low,Profit
3,2011-01-01,45.0,-26.055,1,5,Low,Loss
4,2011-01-01,114.0,37.770,1,5,Medium,Profit


In [7]:
output_path = Path("cleaned_customer_data.csv")

df_clean.to_csv(output_path, index=False)

print("Updated dataset saved successfully!")
print("Final shape:", df_clean.shape)
print("Total columns:", len(df_clean.columns))
print("\nNew feature counts:")
print(df_clean["sales_category"].value_counts())
print("\nProfit status counts:")
print(df_clean["profit_status"].value_counts())

Updated dataset saved successfully!
Final shape: (51290, 28)
Total columns: 28

New feature counts:
sales_category
Low       27918
Medium    16630
High       6742
Name: count, dtype: int64

Profit status counts:
profit_status
Profit        38079
Loss          12543
Break-even      668
Name: count, dtype: int64


In [8]:
def preprocess_customer_data(data):
    data = data.copy()

    data = data.drop_duplicates()

    data["order_date"] = pd.to_datetime(
        data["order_date"], errors="coerce"
    )
    data["ship_date"] = pd.to_datetime(
        data["ship_date"], errors="coerce"
    )

    data["order_month"] = data["order_date"].dt.month
    data["order_day"] = data["order_date"].dt.day
    data["order_day_of_week"] = data["order_date"].dt.dayofweek

    data["sales_category"] = pd.cut(
        data["sales"],
        bins=[-float("inf"), 100, 500, float("inf")],
        labels=["Low", "Medium", "High"]
    )

    data["profit_status"] = data["profit"].apply(
        lambda x: "Profit" if x > 0
        else ("Loss" if x < 0 else "Break-even")
    )

    return data


test_df = preprocess_customer_data(df)

print("Reusable preprocessing function tested!")
print("Input shape:", df.shape)
print("Output shape:", test_df.shape)
print("Missing values:", test_df.isnull().sum().sum())
print("Duplicate rows:", test_df.duplicated().sum())

Reusable preprocessing function tested!
Input shape: (51290, 23)
Output shape: (51290, 28)
Missing values: 0
Duplicate rows: 0


In [9]:
saved_df = pd.read_csv("cleaned_customer_data.csv")

print("Saved dataset loaded successfully!")
print("Dataset shape:", saved_df.shape)
print("\nMissing values:", saved_df.isnull().sum().sum())
print("Duplicate rows:", saved_df.duplicated().sum())
print("\nNew features present:")

for column in [
    "order_month",
    "order_day",
    "order_day_of_week",
    "sales_category",
    "profit_status"
]:
    print(column, ":", column in saved_df.columns)

Saved dataset loaded successfully!
Dataset shape: (51290, 28)

Missing values: 0
Duplicate rows: 0

New features present:
order_month : True
order_day : True
order_day_of_week : True
sales_category : True
profit_status : True
